# P18 · Mini Transformer from Scratch — Indian place names
Runtime ▸ Change runtime type ▸ **T4 GPU**, then Run all. Data: GeoNames India (CC BY 4.0).

In [ ]:
!mkdir -p src data out

In [ ]:
%%writefile src/data.py

import os, re, io, zipfile, random, urllib.request

URL = "https://download.geonames.org/export/dump/IN.zip"   # GeoNames, licence CC BY 4.0

def main(out="data", seed=0):
    os.makedirs(out, exist_ok=True)
    raw = os.path.join(out, "IN.zip")
    if not os.path.exists(raw):
        print("downloading GeoNames India dump ...")
        urllib.request.urlretrieve(URL, raw)
    names, rows = set(), 0
    with zipfile.ZipFile(raw) as z, z.open("IN.txt") as f:
        for line in io.TextIOWrapper(f, encoding="utf-8"):
            p = line.rstrip("\n").split("\t")
            if len(p) < 8 or p[6] != "P":          # feature class P = populated place
                continue
            rows += 1
            n = re.sub(r"[^a-z]+", " ", p[2].lower()).strip()   # asciiname column
            if 3 <= len(n) <= 24:
                names.add(n)
    names = sorted(names)
    random.Random(seed).shuffle(names)
    n = len(names); a, b = int(.9 * n), int(.95 * n)
    for fn, part in [("train", names[:a]), ("val", names[a:b]), ("test", names[b:])]:
        open(f"{out}/{fn}.txt", "w").write("\n".join(part))
    print(f"populated-place rows: {rows:,} | unique clean names: {n:,}")
    print(f"train/val/test = {a:,}/{b-a:,}/{n-b:,}")
    print("sample:", names[:10])

if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/model.py

import math, torch, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 30
    block_size: int = 32
    n_layer: int = 4
    n_head: int = 6
    n_embd: int = 192
    dropout: float = 0.1

def sinusoidal(T, D):
    """Fixed sinusoidal positional encoding (Vaswani et al., 2017)."""
    pos = torch.arange(T).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, D, 2).float() * (-math.log(10000.0) / D))
    pe = torch.zeros(T, D)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.h = c.n_head
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.drop = nn.Dropout(c.dropout)
        # lower-triangular mask: position t may only look at positions <= t
        m = torch.tril(torch.ones(c.block_size, c.block_size)).bool()
        self.register_buffer("mask", m.view(1, 1, c.block_size, c.block_size))

    def forward(self, x, return_att=False):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        att = (q @ k.transpose(-2, -1)) / math.sqrt(C // self.h)
        att = att.masked_fill(~self.mask[:, :, :T, :T], float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (self.drop(att) @ v).transpose(1, 2).reshape(B, T, C)
        y = self.drop(self.proj(y))
        return (y, att) if return_att else y

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = CausalSelfAttention(c)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(),
                                 nn.Linear(4 * c.n_embd, c.n_embd), nn.Dropout(c.dropout))
    def forward(self, x, return_att=False):
        if return_att:
            a, att = self.attn(self.ln1(x), True)
            x = x + a
            return x + self.mlp(self.ln2(x)), att
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.register_buffer("pe", sinusoidal(c.block_size, c.n_embd))
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_att=False):
        B, T = idx.shape
        x = self.drop(self.tok(idx) * math.sqrt(self.c.n_embd) + self.pe[:T])
        atts = []
        for b in self.blocks:
            if return_att:
                x, a = b(x, True); atts.append(a)
            else:
                x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        if return_att:
            return logits, loss, atts
        return logits, loss

    def n_params(self):
        return sum(p.numel() for p in self.parameters())

In [ ]:
%%writefile src/train.py

import os, math, time, json, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from model import Config, MiniGPT

PAD, EOS = 0, 1          # "." in the text view; id 1 is both start and end token
BLOCK = 32

def load(path):
    return open(path).read().split("\n")

def build_vocab(names):
    chars = sorted(set("".join(names)))
    return {c: i + 2 for i, c in enumerate(chars)}

def encode(names, stoi, block=BLOCK):
    out = torch.full((len(names), block + 1), PAD, dtype=torch.long)
    for i, n in enumerate(names):
        s = [EOS] + [stoi[c] for c in n] + [EOS]
        out[i, :len(s)] = torch.tensor(s)
    x, y = out[:, :-1].clone(), out[:, 1:].clone()
    y[y == PAD] = -100
    return x, y

@torch.no_grad()
def evaluate(model, x, y, dev, bs=1024):
    model.eval(); tot, cnt = 0.0, 0
    for i in range(0, len(x), bs):
        xb, yb = x[i:i+bs].to(dev), y[i:i+bs].to(dev)
        logits, _ = model(xb)
        l = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1),
                                              ignore_index=-100, reduction="sum")
        tot += l.item(); cnt += (yb != -100).sum().item()
    model.train()
    return tot / cnt          # nats per character

def main(epochs=6, bs=256, lr=2e-3, out="out"):
    os.makedirs(out, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    tr, va, te = load("data/train.txt"), load("data/val.txt"), load("data/test.txt")
    stoi = build_vocab(tr + va + te)
    xt, yt = encode(tr, stoi); xv, yv = encode(va, stoi); xe, ye = encode(te, stoi)
    cfg = Config(vocab_size=len(stoi) + 2)
    model = MiniGPT(cfg).to(dev)
    print(f"device={dev} params={model.n_params():,} vocab={cfg.vocab_size}")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    steps_per_epoch = math.ceil(len(xt) / bs); total = epochs * steps_per_epoch; warm = 200
    sched = lambda s: lr * (s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    log_steps, tr_l, va_l = [], [], []
    best, step, t0, run = 1e9, 0, time.time(), 0.0
    for ep in range(epochs):
        perm = torch.randperm(len(xt))
        for i in range(0, len(xt), bs):
            ix = perm[i:i+bs]
            xb, yb = xt[ix].to(dev), yt[ix].to(dev)
            for g in opt.param_groups: g["lr"] = sched(step)
            _, loss = model(xb, yb)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            run = 0.9 * run + 0.1 * loss.item() if step else loss.item()
            step += 1
            if step % 200 == 0 or step == total:
                v = evaluate(model, xv, yv, dev)
                log_steps.append(step); tr_l.append(run); va_l.append(v)
                print(f"ep {ep+1} step {step}/{total} train {run:.3f} val {v:.3f} ({v/math.log(2):.3f} bpc)")
                if v < best:
                    best = v
                    torch.save({"model": model.state_dict(), "cfg": cfg.__dict__, "stoi": stoi}, f"{out}/mini_gpt.pt")
    secs = time.time() - t0
    ck = torch.load(f"{out}/mini_gpt.pt", map_location=dev); model.load_state_dict(ck["model"])
    test = evaluate(model, xe, ye, dev)
    m = {"params": model.n_params(), "train_seconds": round(secs, 1), "val_bpc": best / math.log(2),
         "test_bpc": test / math.log(2), "epochs": epochs, "train_names": len(tr)}
    json.dump(m, open(f"{out}/metrics_transformer.json", "w"), indent=2); print(m)
    plt.figure(figsize=(6, 4))
    plt.plot(log_steps, tr_l, label="train (smoothed)"); plt.plot(log_steps, va_l, label="validation")
    plt.xlabel("step"); plt.ylabel("loss (nats/char)"); plt.title("Mini-Transformer training loss")
    plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.savefig(f"{out}/loss_curve.png", dpi=150)

if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/sample.py

import torch
from model import Config, MiniGPT

def load_model(path="out/mini_gpt.pt", dev="cpu"):
    ck = torch.load(path, map_location=dev)
    m = MiniGPT(Config(**ck["cfg"])).to(dev); m.load_state_dict(ck["model"]); m.eval()
    stoi = ck["stoi"]; itos = {i: c for c, i in stoi.items()}
    return m, stoi, itos

@torch.no_grad()
def generate(model, stoi, itos, prefix="", n=10, temperature=0.8, seed=None, max_len=24):
    """Autoregressive sampling. temperature<1 = safer/repetitive, >1 = more random."""
    if seed is not None: torch.manual_seed(seed)
    prefix = prefix.lower().strip()
    bad = [c for c in prefix if c not in stoi]
    if bad: raise ValueError(f"characters not in vocabulary: {bad}")
    dev = next(model.parameters()).device
    start = [1] + [stoi[c] for c in prefix]
    idx = torch.tensor([start] * n, device=dev)
    done = torch.zeros(n, dtype=torch.bool, device=dev)
    for _ in range(max_len - len(prefix)):
        logits, _ = model(idx[:, -model.c.block_size:])
        logits = logits[:, -1, :] / max(temperature, 1e-3)
        logits[:, 0] = float("-inf")                       # never emit PAD
        nxt = torch.multinomial(torch.softmax(logits, -1), 1).squeeze(1)
        nxt[done] = 1
        idx = torch.cat([idx, nxt[:, None]], 1)
        done |= nxt == 1
        if done.all(): break
    out = []
    for row in idx.tolist():
        s = ""
        for t in row[1:]:
            if t == 1: break
            s += itos[t]
        out.append(s.strip().title())
    return out

if __name__ == "__main__":
    m, stoi, itos = load_model()
    for t in (0.5, 0.8, 1.2):
        print(t, generate(m, stoi, itos, n=10, temperature=t, seed=42))

## 1. Download and prepare data (U1)

In [ ]:
!python src/data.py

## 2. Smoke test (30 s): shapes, causal mask, loss at init should be ≈ ln(vocab) ≈ 3.4

In [ ]:
import sys; sys.path.insert(0, "src")
import torch
from model import Config, MiniGPT
m = MiniGPT(Config(vocab_size=30)); x = torch.randint(2, 30, (4, 20))
logits, loss, atts = m(x, x, return_att=True)
print("params", m.n_params(), "| logits", tuple(logits.shape), "| loss", round(loss.item(), 3))
print("future attention is zero:", bool((atts[0][0, 0].triu(1) == 0).all()))

## 3. Train (about 10–15 min on a T4) → `out/loss_curve.png`

In [ ]:
!cd /content && PYTHONPATH=src python src/train.py

## 4. Temperature sampling and the 10 names

In [ ]:
!PYTHONPATH=src python src/sample.py

In [ ]:
from IPython.display import Image; Image("out/loss_curve.png")

## 5. Download results for repo and report

In [ ]:
!zip -r p18_results.zip src out/loss_curve.png out/metrics_transformer.json out/mini_gpt.pt
from google.colab import files; files.download("p18_results.zip")